# Complaint Triage — Exploratory Analysis

This notebook is for poking around the results interactively, beyond what `src/evaluate.py` prints. Run `python src/classify.py` first so `data/results.csv` exists.

Everything here reads `data/results.csv` — it doesn't call the Anthropic API, so it's free and instant to re-run.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Let us import from src/ (taxonomy.py etc.) if we want it later
sys.path.insert(0, str(Path.cwd().parent / "src"))

ROOT = Path.cwd().parent
RESULTS_FILE = ROOT / "data" / "results.csv"

df = pd.read_csv(RESULTS_FILE)
# "correct" comes back from CSV as the string "True"/"False", not a real bool
df["correct"] = df["correct"].astype(str).map({"True": True, "False": False})

print(f"{len(df)} rows loaded")
df.head()

## Overall accuracy

We exclude `PARSE_ERROR` rows from the accuracy calculation — those are cases where the model's response couldn't be parsed as JSON at all, which is a different kind of failure than "picked the wrong category."

In [ ]:
scored = df[df["predicted_product"] != "PARSE_ERROR"]
parse_errors = len(df) - len(scored)

accuracy = scored["correct"].mean()
print(f"Accuracy: {accuracy:.1%}  ({scored['correct'].sum()}/{len(scored)} correct)")
print(f"Parse errors: {parse_errors}")

## Per-category accuracy

Which product categories does the model nail, and which does it struggle with? Sorted worst to best so the problem areas are at the top.

In [ ]:
per_category = (
    scored.groupby("true_product")["correct"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "accuracy", "count": "n"})
    .sort_values("accuracy")
)
per_category

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=per_category["accuracy"], y=per_category.index, color="#4C72B0")
plt.xlabel("Accuracy")
plt.ylabel("")
plt.xlim(0, 1)
plt.title("Per-category accuracy")
plt.tight_layout()
plt.show()

## Reading actual misclassifications

Numbers only tell you *that* the model is confused — reading the actual narratives tells you *why*. This is the same idea as the "Failures" tab in the Streamlit app, but easier to slice and filter here.

In [ ]:
wrong = df[(df["correct"] == False) & (df["predicted_product"] != "PARSE_ERROR")]

# Example: look at cases where the true label was "Debt collection"
# but the model guessed something else. Change this filter to explore
# other categories.
subset = wrong[wrong["true_product"] == "Debt collection"]

for _, row in subset.head(5).iterrows():
    print(f"#{row['complaint_id']}  true={row['true_product']!r}  predicted={row['predicted_product']!r}")
    print(f"  narrative: {row['narrative'][:300]}...")
    print(f"  reason:    {row['reason']}")
    print()

## Most common confusion pairs

Which two categories does the model mix up most often? A high count here usually means the categories are genuinely ambiguous from the narrative text alone (e.g. a complaint about inaccurate information on a debt collection account could reasonably be filed under either "Debt collection" or "Credit reporting").

In [ ]:
from collections import Counter

pairs = Counter(zip(wrong["true_product"], wrong["predicted_product"]))
for (true_p, pred_p), count in pairs.most_common(10):
    print(f"{count:>3}x   {true_p}  ->  {pred_p}")

## Urgency ratings

The model also assigns an urgency level (low/medium/high) to every complaint. This wasn't part of the CFPB's ground truth, so we can't score it for "correctness" — but it's worth sanity-checking the distribution isn't just "everything is high".

In [ ]:
scored["urgency"].value_counts().reindex(["low", "medium", "high"]).fillna(0)